# Notebook 05 — Assembly & Validation
Merge real + TabSyn + GReaT into the final **1.2M session HoneySynth-1M** dataset.
Run full quality validation (Wasserstein, adversarial AUC, TSTR) before training.

| Source | Target | Actual |
|--------|--------|--------|
| Real Cowrie | 180,000 (15%) | from Notebook 01/02 |
| Real Transfer (CIC/UNSW) | 60,000 (5%) | from Notebook 01/02 |
| TabSyn synthetic | 720,000 (60%) | from Notebook 03 |
| GReaT synthetic | 240,000 (20%) | from Notebook 04 |
| **Total** | **1,200,000** | |

In [1]:
import sys, os, logging, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger()

ROOT       = Path("..").resolve()
DATA_PROC  = ROOT / "data" / "processed"
DATA_SYNTH = ROOT / "data" / "synthetic"
# data/final/ is FROZEN: every published number depends on it and the old
# arrays must survive until the new ones are measured against them
# (DATA_INTEGRITY.md section 6). The rebuild writes alongside it.
DATA_FINAL = ROOT / "data" / "final_v2"
DATA_FINAL.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

from configs.schema import N_FEATURES, N_CLASSES, IDX_TO_LABEL, SPLIT_TARGETS
from src.extractors.temporal import TEMPORAL_FEATURE_NAMES

## 5.1 Load All Sources

In [2]:
X_real   = np.load(DATA_PROC  / "X_real.npy")
y_real   = np.load(DATA_PROC  / "y_real.npy")

# ---- Split BENIGN (label 45) out of the 45-class problem ------------------
# Benign is not a kill-chain state, so it is not a 46th class: it is saved
# separately for the two-stage gate (DATA_INTEGRITY.md section 5, which
# prefers a gate precisely because benign has no phase and no DAG edges).
#
# READ THIS BEFORE USING IT: real honeypot traffic contains NO benign
# sessions, so every benign row here comes from CIC/UNSW netflow. A gate
# trained on this can learn "netflow-shaped = benign" -- the data source,
# not the concept -- and will score well doing it. Do not report it as
# benign detection without that caveat.
BENIGN_IDX = 45
benign_mask = (y_real == BENIGN_IDX)
if benign_mask.any():
    np.save(DATA_PROC / "X_benign.npy", X_real[benign_mask])
    np.save(DATA_PROC / "y_benign.npy", y_real[benign_mask])
    print(f"Benign split off: {int(benign_mask.sum()):,} rows "
          f"-> X_benign.npy (NOT part of the 45-class set)")
    X_real, y_real = X_real[~benign_mask], y_real[~benign_mask]

# ---- TabSyn and GReaT are both OPTIONAL -----------------------------------
# TabSyn existed to multiply 15,000 synthetic Cowrie sessions into 720,000.
# That job now belongs to cowrie_simulator (notebook 01), which generates
# events through the real parser so its derived features are computed, not
# drawn. TabSyn still generates derived features alongside the primitives
# they depend on, which is what made 47% of the frozen synthetic rows
# arithmetically impossible -- and Part 2 of DATA_INTEGRITY.md section 4,
# which fixes that, is NOT done. So a build with TabSyn present reproduces
# the incoherence; a build without it does not.
TABSYN_PATH = DATA_SYNTH / "X_tabsyn.npy"
# OPT-IN, not file-existence. The August X_tabsyn.npy is still on disk and
# is 720,000 rows -- on the first smoke build it silently became 98.6% of
# the dataset, which is precisely the data the rebuild exists to replace.
# Set HS_INCLUDE_TABSYN=1 only once Part 2 (primitives-only generation) has
# landed; until then those rows break their own log1p/iat identities.
INCLUDE_TABSYN = TABSYN_PATH.exists() and os.environ.get("HS_INCLUDE_TABSYN") == "1"
if INCLUDE_TABSYN:
    X_tabsyn = np.load(TABSYN_PATH)
    y_tabsyn = np.load(DATA_SYNTH / "y_tabsyn.npy")
    print("WARNING: X_tabsyn.npy found and will be INCLUDED. Unless Part 2 has "
          "landed, ~40% of these rows break their own log1p/iat identities.")
else:
    X_tabsyn = np.empty((0, X_real.shape[1]), dtype=X_real.dtype)
    y_tabsyn = np.empty((0,), dtype=y_real.dtype)
    print("NOTE: X_tabsyn.npy not found -> assembling without TabSyn.")

GREAT_PATH = DATA_SYNTH / "X_great.npy"
INCLUDE_GREAT = GREAT_PATH.exists()
if INCLUDE_GREAT:
    X_great = np.load(GREAT_PATH)
    y_great = np.load(DATA_SYNTH / "y_great.npy")
else:
    X_great = np.empty((0, X_real.shape[1]), dtype=X_real.dtype)
    y_great = np.empty((0,), dtype=y_real.dtype)
    print("NOTE: X_great.npy not found -> GReaT skipped.")

print("\nLoaded sources:")
print(f"  Real+sim: X={X_real.shape}    y={y_real.shape}")
print(f"  TabSyn:   X={X_tabsyn.shape}  (included={INCLUDE_TABSYN})")
print(f"  GReaT:    X={X_great.shape}   (included={INCLUDE_GREAT})")
print(f"\n  TOTAL: {len(X_real)+len(X_tabsyn)+len(X_great):,} sessions")


Benign split off: 193,000 rows -> X_benign.npy (NOT part of the 45-class set)


NOTE: X_tabsyn.npy not found -> assembling without TabSyn.
NOTE: X_great.npy not found -> GReaT skipped.

Loaded sources:
  Real+sim: X=(1556809, 128)    y=(1556809,)
  TabSyn:   X=(0, 128)  (included=False)
  GReaT:    X=(0, 128)   (included=False)

  TOTAL: 1,556,809 sessions


## 5.2 Resample to Exact Split Targets

In [3]:
from sklearn.utils import resample

def resample_to_target(X, y, target_n, seed=42):
    """Returns (X, y, idx). `idx` is what makes per-row provenance survive the
    resample -- cell 7 needs it to label each row by its true source instead of
    assuming the first len(X_real) rows are real."""
    n = len(X)
    if n == target_n:
        idx = np.arange(n)
    else:
        idx = np.random.RandomState(seed).choice(n, target_n, replace=(n < target_n))
    return X[idx], y[idx], idx

# Keep EVERY row notebook 01 produced. The old `real_cowrie + real_transfer`
# target of 240,000 was sized for the 1.2M plan where TabSyn supplied 720,000;
# applying it now would resample ~1.5M rows (114k real Cowrie + 722k netflow +
# 720k generated) down to 240,000 and throw the rebuild away.
real_target = len(X_real)
X_real_f, y_real_f, idx_real = resample_to_target(X_real, y_real, real_target, seed=1)

if INCLUDE_TABSYN:
    X_tabsyn_f, y_tabsyn_f, _ = resample_to_target(
        X_tabsyn, y_tabsyn, SPLIT_TARGETS["tabsyn_synth"], seed=2)
else:
    X_tabsyn_f = np.empty((0, X_real.shape[1]), dtype=np.float32)
    y_tabsyn_f = np.empty((0,), dtype=np.int64)

if INCLUDE_GREAT:
    X_great_f, y_great_f, _ = resample_to_target(
        X_great, y_great, SPLIT_TARGETS["great_synth"], seed=3)
else:
    X_great_f = np.empty((0, X_real.shape[1]), dtype=np.float32)
    y_great_f = np.empty((0,), dtype=np.int64)

print("Resampled to targets:")
print(f"  Real+sim: {len(X_real_f):,} (kept all)")
print(f"  TabSyn:   {len(X_tabsyn_f):,}" + ("" if INCLUDE_TABSYN else " (skipped)"))
print(f"  GReaT:    {len(X_great_f):,}" + ("" if INCLUDE_GREAT else " (skipped)"))


Resampled to targets:
  Real+sim: 1,556,809 (kept all)
  TabSyn:   0 (skipped)
  GReaT:    0 (skipped)


## 5.3 Merge + Source Tags

In [4]:
X_full = np.vstack([X_real_f, X_tabsyn_f, X_great_f]).astype(np.float32)
y_full = np.concatenate([y_real_f, y_tabsyn_f, y_great_f]).astype(np.int64)

# Per-row provenance, read from real_meta.parquet rather than assumed from
# position. The old version hard-coded ["real"] * len(X_real_f) and
# ["cowrie"] * len(X_real_f), which was wrong twice over: 76% of those rows are
# CIC/UNSW netflow, not Cowrie, and X_real now also holds generated
# `simulated_cowrie_events` rows that are not real at all. Reporting per source
# is the one decision that stops a headline number hiding what it is built from
# (DATA_INTEGRITY.md section 5, item 6).
meta = pd.read_parquet(DATA_PROC / "real_meta.parquet")
assert len(meta) == len(X_real) + int(benign_mask.sum()), (
    f"real_meta.parquet has {len(meta)} rows but X_real had "
    f"{len(X_real) + int(benign_mask.sum())} before the benign split -- "
    f"rerun notebook 02 so metadata and features line up")
src_real_all = meta["source"].to_numpy()
if benign_mask.any():                      # same rows the benign split removed
    src_real_all = src_real_all[~benign_mask]

generator_full = np.concatenate([
    src_real_all[idx_real],
    np.full(len(X_tabsyn_f), "tabsyn"),
    np.full(len(X_great_f), "great"),
])
# "real" means measured, not generated. cowrie_real / cic_transfer /
# unsw_transfer are measured; simulated_cowrie_events, tabsyn and great are not.
REAL_SOURCES = {"cowrie_real", "cic_transfer", "unsw_transfer"}
source_full = np.where(np.isin(generator_full, list(REAL_SOURCES)),
                       "real", "synthetic")

print(f"Merged dataset: X={X_full.shape}  y={y_full.shape}")
assert len(generator_full) == len(X_full)
print("\nRows by generator:")
for g, n in zip(*np.unique(generator_full, return_counts=True)):
    print(f"  {g:<28} {n:>9,}")
print("Rows by provenance:")
for s, n in zip(*np.unique(source_full, return_counts=True)):
    print(f"  {s:<28} {n:>9,}  ({100*n/len(X_full):.1f}%)")

# Shuffle once (keep alignment across all arrays)
rng = np.random.RandomState(42)
perm = rng.permutation(len(X_full))
X_full, y_full = X_full[perm], y_full[perm]
source_full, generator_full = source_full[perm], generator_full[perm]


Merged dataset: X=(1556809, 128)  y=(1556809,)

Rows by generator:
  cic_transfer                   557,646
  cowrie_real                    114,490
  simulated_cowrie_events        720,000
  unsw_transfer                  164,673
Rows by provenance:


  real                           836,809  (53.8%)
  synthetic                      720,000  (46.2%)


## 5.4 Stratified Split — Train / Val / Test(synth) / Test(real, TSTR)

In [5]:
def safe_stratified_split(X, y, test_size, seed=42):
    counts = np.bincount(y, minlength=N_CLASSES)
    can_stratify = bool((counts[counts > 0] >= 2).all())
    return train_test_split(
        X, y, test_size=test_size,
        stratify=y if can_stratify else None,
        random_state=seed,
    )

real_mask  = source_full == "real"
synth_mask = source_full == "synthetic"

X_r, y_r = X_full[real_mask],  y_full[real_mask]
X_s, y_s = X_full[synth_mask], y_full[synth_mask]

# Held-out REAL test set → TSTR evaluation (the paper's key metric)
n_test_real = min(60_000, int(len(X_r) * 0.30))
X_r_train, X_test_real, y_r_train, y_test_real = safe_stratified_split(
    X_r, y_r, test_size=n_test_real, seed=11)

# Held-out SYNTHETIC test set → standard evaluation
n_test_synth = min(60_000, int(len(X_s) * 0.30))
X_s_train, X_test_synth, y_s_train, y_test_synth = safe_stratified_split(
    X_s, y_s, test_size=n_test_synth, seed=12)

# Combine remaining for train/val pool
X_pool = np.vstack([X_r_train, X_s_train])
y_pool = np.concatenate([y_r_train, y_s_train])

X_train, X_val, y_train, y_val = safe_stratified_split(
    X_pool, y_pool, test_size=0.10, seed=13)

# ---- carry the per-row GENERATOR through the same partition ---------------
# Without this the shipped dataset cannot say which rows are synthetic.
# `prov_<split>.npy` distinguishes measured-vs-imputed temporal evidence, but
# BOTH cowrie_real and simulated_cowrie_events are `event_stream`, so provenance
# alone cannot separate real SSH sessions from generated ones -- which makes any
# real-vs-synthetic analysis irreproducible from the artifact.
#
# The index arrays are split with the SAME function, seeds, test sizes and
# stratify targets as the feature arrays above. train_test_split is
# deterministic given those, so the partition is identical and the existing
# X_*/y_* arrays are untouched -- this only recovers the alignment.
_ir = np.where(real_mask)[0].reshape(-1, 1)
_is = np.where(synth_mask)[0].reshape(-1, 1)
_ir_tr, _ir_te, _, _ = safe_stratified_split(_ir, y_r, test_size=n_test_real, seed=11)
_is_tr, _is_te, _, _ = safe_stratified_split(_is, y_s, test_size=n_test_synth, seed=12)
_pool_idx = np.vstack([_ir_tr, _is_tr])
_tr_idx, _val_idx, _, _ = safe_stratified_split(_pool_idx, y_pool, test_size=0.10, seed=13)

gen_splits = {
    "train":      generator_full[_tr_idx.ravel()],
    "val":        generator_full[_val_idx.ravel()],
    "test_real":  generator_full[_ir_te.ravel()],
    "test_synth": generator_full[_is_te.ravel()],
}
for _k, _v in gen_splits.items():
    assert len(_v) == {"train": len(y_train), "val": len(y_val),
                       "test_real": len(y_test_real),
                       "test_synth": len(y_test_synth)}[_k], \
        f"generator alignment for {_k} does not match its split length"
print("\nRows by generator, per split:")
print(pd.DataFrame({k: pd.Series(v).value_counts() for k, v in gen_splits.items()})
      .fillna(0).astype(int).to_string())

print("Final splits:")
print(f"  Train:        {len(X_train):>9,}")
print(f"  Val:          {len(X_val):>9,}")
print(f"  Test (synth): {len(X_test_synth):>9,}")
print(f"  Test (real):  {len(X_test_real):>9,}  ← TSTR evaluation")
print(f"  {'─'*30}")
print(f"  TOTAL:        {len(X_train)+len(X_val)+len(X_test_synth)+len(X_test_real):>9,}")


Rows by generator, per split:


                          train    val  test_real  test_synth
cic_transfer             465835  51825      39986           0
cowrie_real               95652  10671       8167           0
simulated_cowrie_events  594031  65969          0       60000
unsw_transfer            137610  15216      11847           0
Final splits:
  Train:        1,293,128
  Val:            143,681
  Test (synth):    60,000
  Test (real):     60,000  ← TSTR evaluation
  ──────────────────────────────
  TOTAL:        1,556,809


In [6]:
# ── Temporal imputation + availability mask (Part 0, decided 2026-11-24) ────
# `extract_temporal` now returns NaN for anything a session cannot support, so
# 46.7% of rows carry NaN in the 24-feature temporal block. MT3's LSTM branch
# cannot consume NaN, so a choice has to be made -- and the whole point of Part 0
# is that it is made HERE, in the open, instead of by a `nan_to_num` call.
#
# CHOSEN: impute, and ship the availability mask alongside.
#
# Rejected alternative: train on `event_stream` rows only. It keeps all 45 attack
# classes above the 2,000 floor (min class 16,000) but drops BENIGN entirely --
# benign exists only in netflow and so has ZERO event-stream rows -- and shrinks
# test_real from 60,000 to 8,167.
#
# Three rules make this defensible rather than a quiet refill:
#
#  1. The fill value is the per-column median of rows that ACTUALLY MEASURED the
#     feature (`event_stream`), never of the whole column. A median over imputed
#     rows would drift toward whatever was already filled in.
#  2. It is computed on X_train ONLY and applied to val/test, so the fill value
#     carries no information from the evaluation splits.
#  3. The mask ships with the data. `M_*.npy` is 1 where the feature was
#     measured and 0 where it was filled, and `prov_*.npy` carries the per-row
#     provenance level. Nothing downstream has to guess which is which, and
#     results MUST be reported split by provenance (DATA_INTEGRITY.md 5 item 6,
#     extended by one axis).
#
# N_FEATURES stays 128: the mask is metadata, not a 129th feature, so the live
# pipeline and the model output shapes are untouched.
from src.validators.feature_integrity import provenance_from_nan

N_TEMPORAL = 24
_splits_pre = {
    "train":      (X_train, y_train),
    "val":        (X_val, y_val),
    "test_real":  (X_test_real, y_test_real),
    "test_synth": (X_test_synth, y_test_synth),
}

# masks and provenance captured BEFORE imputing -- afterwards the NaN signature
# is gone and provenance_from_nan can no longer recover anything.
masks = {k: (~np.isnan(X[:, :N_TEMPORAL])).astype(np.uint8)
         for k, (X, _) in _splits_pre.items()}
provs = {k: provenance_from_nan(X) for k, (X, _) in _splits_pre.items()}

# rule 1 + 2: medians from measured TRAIN rows only
_measured = provs["train"] == "event_stream"
assert _measured.any(), "no event_stream rows in train -- cannot derive a fill value"
temporal_medians = np.nanmedian(X_train[_measured][:, :N_TEMPORAL], axis=0)
assert np.isfinite(temporal_medians).all(), "a temporal median came out non-finite"

_TI = {n: j for j, n in enumerate(TEMPORAL_FEATURE_NAMES)}


def _repair_order_stats(blk, measured):
    """Make imputed order statistics coherent with the MEASURED ones in the row.

    A per-column median fill is independent per column, so it happily produces
    `iat_median` outside the row's own measured `[iat_min, iat_max]`. Measured
    on the first attempt: 18.1% of rows, and 99.9% of those had the median
    imputed while min and max were real. That is the arithmetically-impossible
    row this whole effort exists to eliminate, reintroduced by the fix for it.
    So imputed values are clipped into the window the measurements allow.

    Only IMPUTED entries move. A measured value is never adjusted to accommodate
    a filled one -- the measurement is the ground truth and the fill is the
    guess.
    """
    inf = np.inf
    mn, me, md, mx = (_TI["iat_min"], _TI["iat_mean"],
                      _TI["iat_median"], _TI["iat_max"])
    # the window the row's own measurements permit
    lo = np.where(measured[:, mn], blk[:, mn], -inf)
    hi = np.where(measured[:, mx], blk[:, mx], inf)

    # widen an imputed min / max so it cannot contradict a measured interior
    inner_lo = np.full(len(blk), inf)
    inner_hi = np.full(len(blk), -inf)
    for c in (me, md):
        inner_lo = np.where(measured[:, c], np.minimum(inner_lo, blk[:, c]), inner_lo)
        inner_hi = np.where(measured[:, c], np.maximum(inner_hi, blk[:, c]), inner_hi)
    fix = ~measured[:, mn] & np.isfinite(inner_lo)
    blk[fix, mn] = np.minimum(blk[fix, mn], inner_lo[fix])
    fix = ~measured[:, mx] & np.isfinite(inner_hi)
    blk[fix, mx] = np.maximum(blk[fix, mx], inner_hi[fix])

    # recompute the window after those moves, then clip the imputed interior
    lo = np.where(measured[:, mn], blk[:, mn], blk[:, mn])
    hi = np.where(measured[:, mx], blk[:, mx], blk[:, mx])
    lo, hi = np.minimum(lo, hi), np.maximum(lo, hi)
    for c in (me, md):
        fix = ~measured[:, c]
        blk[fix, c] = np.clip(blk[fix, c], lo[fix], hi[fix])

    # iat_cv is a pure function of std and mean; if either was filled, the
    # identity must be restored or the row claims a cv it does not have
    cv, sd = _TI["iat_cv"], _TI["iat_std"]
    touched = ~measured[:, cv] | ~measured[:, sd] | ~measured[:, me]
    with np.errstate(divide="ignore", invalid="ignore"):
        recomputed = blk[:, sd] / np.maximum(blk[:, me], 1e-9)
    blk[touched, cv] = recomputed[touched]
    return blk


n_filled, n_repaired = {}, {}
for k, (X, _) in _splits_pre.items():
    blk = X[:, :N_TEMPORAL]
    bad = np.isnan(blk)
    n_filled[k] = int(bad.sum())
    blk[bad] = np.take(temporal_medians, np.where(bad)[1])
    before = blk.copy()
    blk = _repair_order_stats(blk, masks[k].astype(bool))
    n_repaired[k] = int((np.abs(blk - before) > 1e-12).sum())
    X[:, :N_TEMPORAL] = blk
    assert not np.isnan(X).any(), f"{k} still carries NaN after imputation"
    # the repair must leave the order statistics coherent
    _mn, _me, _md, _mx = (blk[:, _TI["iat_min"]], blk[:, _TI["iat_mean"]],
                          blk[:, _TI["iat_median"]], blk[:, _TI["iat_max"]])
    for nm, v in (("mean", _me), ("median", _md)):
        n_bad = int(((v < _mn - 1e-6) | (v > _mx + 1e-6)).sum())
        assert n_bad == 0, f"{k}: {n_bad} rows have iat_{nm} outside [min, max]"

print("Temporal imputation (median of measured train rows):")
for k in _splits_pre:
    p = pd.Series(provs[k]).value_counts()
    meas = 100.0 * (provs[k] == "event_stream").mean()
    print(f"  {k:<11} n={len(provs[k]):>9,}  measured {meas:>5.1f}%  "
          f"cells filled {n_filled[k]:>10,}  repaired {n_repaired[k]:>9,}")
print("\nProvenance per split:")
print(pd.DataFrame({k: pd.Series(provs[k]).value_counts() for k in _splits_pre})
      .fillna(0).astype(int).to_string())


2026-11-24 20:33:42,086 INFO TI cache: 100 entries loaded


Temporal imputation (median of measured train rows):
  train       n=1,293,128  measured  53.3%  cells filled  6,585,889  repaired   457,920
  val         n=  143,681  measured  53.3%  cells filled    731,418  repaired    50,750
  test_real   n=   60,000  measured  13.6%  cells filled    565,805  repaired    39,362
  test_synth  n=   60,000  measured 100.0%  cells filled          0  repaired         0

Provenance per split:


                 train    val  test_real  test_synth
event_stream    689682  76640       8167       60000
flow_mean_only  137411  15192      11825           0
flow_stats      465678  51801      39973           0
none               357     48         35           0


## 5.5 Normalise — Fit on Train Only

In [7]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_s      = scaler.transform(X_train).astype(np.float32)
X_val_s        = scaler.transform(X_val).astype(np.float32)
X_test_synth_s = scaler.transform(X_test_synth).astype(np.float32)
X_test_real_s  = scaler.transform(X_test_real).astype(np.float32)

joblib.dump(scaler, DATA_FINAL / "feature_scaler.pkl")
print(f"Scaler fitted on {len(X_train):,} training rows.")
print(f"Mean range: [{scaler.mean_.min():.3f}, {scaler.mean_.max():.3f}]")
print(f"Scale range: [{scaler.scale_.min():.3f}, {scaler.scale_.max():.3f}]")

Scaler fitted on 1,293,128 training rows.
Mean range: [-0.193, 7761.579]
Scale range: [0.000, 879385.041]


## 5.6 Save Final Splits

In [8]:
splits = {
    "X_train": X_train_s,      "y_train": y_train,
    "X_val":   X_val_s,        "y_val":   y_val,
    "X_test_synth": X_test_synth_s, "y_test_synth": y_test_synth,
    "X_test_real":  X_test_real_s,  "y_test_real":  y_test_real,
}
for name, arr in splits.items():
    np.save(DATA_FINAL / f"{name}.npy", arr)
    print(f"  Saved {name}.npy  {arr.shape}")

print(f"\n✓ All splits saved to {DATA_FINAL}")

# Availability masks and provenance ship WITH the splits. Without them nobody
# downstream can tell a measured temporal feature from an imputed one, which
# would re-create exactly the ambiguity Part 0 removed.
for _k in masks:
    np.save(DATA_FINAL / f"M_{_k}.npy", masks[_k])
    np.save(DATA_FINAL / f"prov_{_k}.npy", provs[_k].astype("U14"))
    np.save(DATA_FINAL / f"gen_{_k}.npy", gen_splits[_k].astype("U26"))
    print(f"  Saved M_{_k}.npy {masks[_k].shape}  prov_{_k}.npy  gen_{_k}.npy")

_manifest = {
    "temporal_imputation": {
        "strategy": "per-column median of X_train rows with provenance "
                    "event_stream (i.e. rows that measured the feature)",
        "applied_to": "features 0:24 (A_temporal) of every split",
        "fitted_on": "X_train only, to keep val/test free of the fill value",
        "cells_filled": n_filled,
        "cells_repaired": n_repaired,
        "order_repair": "imputed iat_min/mean/median/max are clipped into "
                        "the window the row's MEASURED order statistics "
                        "allow, and iat_cv is recomputed when std or mean "
                        "was filled. Without this, independent per-column "
                        "medians put iat_median outside [min, max] on 18.1% "
                        "of rows -- the impossible row this work removes, "
                        "reintroduced by the fix for it. Measured values are "
                        "never moved to accommodate a fill.",
        "medians": {TEMPORAL_FEATURE_NAMES[i]: float(temporal_medians[i])
                    for i in range(N_TEMPORAL)},
        "mask_files": "M_<split>.npy  1 = measured, 0 = imputed",
        "provenance_files": "prov_<split>.npy  per-row provenance level",
        "why": "extract_temporal returns NaN for features a session cannot "
               "support; MT3 cannot consume NaN. See DATA_INTEGRITY.md 10e.",
        "reporting_requirement": "any metric computed on these arrays must be "
                                 "reported split by provenance level",
    }
}
with open(DATA_FINAL / "imputation_manifest.json", "w") as _fh:
    json.dump(_manifest, _fh, indent=2)
print(f"  Saved imputation_manifest.json")


  Saved X_train.npy  (1293128, 128)
  Saved y_train.npy  (1293128,)
  Saved X_val.npy  (143681, 128)
  Saved y_val.npy  (143681,)
  Saved X_test_synth.npy  (60000, 128)
  Saved y_test_synth.npy  (60000,)
  Saved X_test_real.npy  (60000, 128)
  Saved y_test_real.npy  (60000,)

✓ All splits saved to C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\final_v2


  Saved M_train.npy (1293128, 24)  prov_train.npy  gen_train.npy
  Saved M_val.npy (143681, 24)  prov_val.npy  gen_val.npy
  Saved M_test_real.npy (60000, 24)  prov_test_real.npy  gen_test_real.npy
  Saved M_test_synth.npy (60000, 24)  prov_test_synth.npy  gen_test_synth.npy
  Saved imputation_manifest.json


## 5.7 Full Quality Validation
This produces every number for **Table 1** of the paper.

In [9]:
from src.validators.quality_checks import run_full_validation

# Use UNSCALED data for Wasserstein/adversarial checks (interpretable units)
report = run_full_validation(
    X_real=X_r_train[:30_000],  y_real=y_r_train[:30_000],
    X_synth=X_s_train[:30_000], y_synth=y_s_train[:30_000],
    output_path=DATA_FINAL / "quality_report.json",
)

print(f"\n{'='*50}")
print(f"VERDICT: {report['verdict']}")
print(f"{'='*50}")

2026-11-24 20:33:48,838 INFO ============================================================


2026-11-24 20:33:48,838 INFO DATASET QUALITY VALIDATION — HoneySynth-1M


2026-11-24 20:33:48,839 INFO ============================================================


2026-11-24 20:33:48,861 INFO Shape check ✗ FAIL: ['NaN values: 281635']


2026-11-24 20:33:48,864 INFO Balance check ⚠ WARN — 39 under-represented classes | min=604 max=12065 ratio=19.9


2026-11-24 20:33:49,052 INFO   ⚠ A_temporal           mean_W=nan  max_W=nan


... (11 outputs left out)

2026-11-24 20:34:06,360 INFO Adversarial AUC: 0.9981


2026-11-24 20:34:06,360 INFO ============================================================


2026-11-24 20:34:06,363 INFO Report saved → C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\final_v2\quality_report.json



VERDICT: NEEDS FIXES — see issues above


## 5.8 Kill-Chain Violation Rate (KCVR) Baseline

In [10]:
from src.generators.kill_chain_simulator import compute_kcvr

# Load metadata with sequences for KCVR computation
meta_frames = []
real_meta = DATA_PROC / "real_meta.parquet"
tabsyn_meta = DATA_SYNTH / "tabsyn_meta.parquet"
great_meta = DATA_SYNTH / "great_meta.parquet"

for p, seq_col in [(tabsyn_meta, "micro_state_sequence" if False else None),
                    (great_meta, "micro_state_sequence")]:
    if p.exists():
        df_m = pd.read_parquet(p)
        if "micro_state_sequence" in df_m.columns:
            meta_frames.append(df_m)

if meta_frames:
    df_seq = pd.concat(meta_frames, ignore_index=True)
    kcvr = compute_kcvr(df_seq, seq_col="micro_state_sequence")
    print(f"Dataset Kill-Chain Violation Rate (post-filter): {kcvr:.4f}")
    print(f"  (Target: 0.0000 — all generated sequences are DAG-valid)")
else:
    print("No sequence metadata found for KCVR computation")

No sequence metadata found for KCVR computation


## 5.9 Dataset Card Summary — Table 1 for Paper

In [11]:
from configs.schema import N_FEATURES, N_CLASSES, FEATURE_GROUPS

summary = {
    "dataset_name":          "HoneySynth-1M",
    "total_sessions":        len(X_train)+len(X_val)+len(X_test_synth)+len(X_test_real),
    "real_sessions":         int(real_mask.sum()),
    "real_pct":              round(real_mask.sum()/len(X_full)*100, 1),
    "tabsyn_sessions":       len(X_tabsyn_f),
    "tabsyn_pct":            round(len(X_tabsyn_f)/len(X_full)*100, 1),
    "great_sessions":        len(X_great_f),
    "great_pct":             round(len(X_great_f)/len(X_full)*100, 1),
    "n_features":            N_FEATURES,
    "n_classes":             N_CLASSES,
    "n_feature_groups":      len(FEATURE_GROUPS),
    "mitre_techniques":      45,
    "mitre_tactics":         9,
    "min_samples_per_class": int(np.bincount(y_full, minlength=N_CLASSES).min()),
    "max_samples_per_class": int(np.bincount(y_full, minlength=N_CLASSES).max()),
    "adversarial_auc":       report["adversarial_auc"]["adversarial_auc"],
    "tstr_macro_f1":         report["tstr"]["tstr_macro_f1"],
    "privacy_note":          "DP-SGD recommended for public release (epsilon=10, delta=1e-6)",
}

print("="*60)
print("  TABLE 1 — Dataset Statistics (HoneySynth-1M)")
print("="*60)
for k, v in summary.items():
    print(f"  {k:<25} {v}")
print("="*60)

with open(DATA_FINAL / "dataset_card.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nDataset card saved → {DATA_FINAL}/dataset_card.json")
print("\n✓ DATASET ASSEMBLY COMPLETE — ready for Phase 5 (model training)")

  TABLE 1 — Dataset Statistics (HoneySynth-1M)
  dataset_name              HoneySynth-1M
  total_sessions            1556809
  real_sessions             836809
  real_pct                  53.8
  tabsyn_sessions           0
  tabsyn_pct                0.0
  great_sessions            0
  great_pct                 0.0
  n_features                128
  n_classes                 45
  n_feature_groups          6
  mitre_techniques          45
  mitre_tactics             9
  min_samples_per_class     16000
  max_samples_per_class     338616
  adversarial_auc           0.9981
  tstr_macro_f1             0.035
  privacy_note              DP-SGD recommended for public release (epsilon=10, delta=1e-6)

Dataset card saved → C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\final_v2/dataset_card.json

✓ DATASET ASSEMBLY COMPLETE — ready for Phase 5 (model training)
